# 🎯 Objetivos

* ⚙️ **Configuración inicial**. Rutas, paquetes y funciones.
* ♻️ **Carga de datos**.
* 📊 **Resumen cuantitativo de resultados**.
* 👤 **Evaluación cualitativa**.

# ⚙️ Configuración

## Rutas (paths)

Seteamos las rutas que vamos a necesitar

In [1]:
import os

# Rutas en función de la carpeta raíz del proyecto (README.md)
base_path = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in locals() else os.path.abspath("..")

# Subrutas
data_dir = os.path.join(base_path, "data")
db_path = os.path.join(data_dir, "steam.db")
src_dir = os.path.join(base_path, "src")
models_dir = os.path.join(base_path, "models")
models_content_dir = os.path.join(models_dir, "content")
models_collab_dir = os.path.join(models_dir, "collaborative")
results_dir = os.path.join(base_path, "res")

## Paquetes y funciones

In [2]:
# Sistema y configuración
import warnings
warnings.filterwarnings("ignore")
import sqlite3
from scipy import sparse

# Modelos
import joblib

# Análisis y manipulación de datos
import pandas as pd
import numpy as np

# Funciones personalizadas    

from utils.recommendation_functions import (
    recommend_by_popularity,
    recommend_by_content_user,
    recommend_by_collaborative,
    recommend_by_hybrid,
)

# ♻️ Carga de datos

## Resultados de los modelos

In [3]:
# k=10
popular_k10 = pd.read_csv(os.path.join(results_dir, "metrics_popularity_LMO_k10_20251130.csv"))
content_k10 = pd.read_csv(os.path.join(results_dir, "metrics_content_LMO_k10_20251130.csv"))
collab_k10 = pd.read_csv(os.path.join(results_dir, "metrics_collaborative_k10_20251130.csv"))
hybrid_k10 = pd.read_csv(os.path.join(results_dir, "metrics_hybrid_k10_a0.2_20251130.csv"))

# k=50
popular_k50 = pd.read_csv(os.path.join(results_dir, "metrics_popularity_LMO_k50_20251130.csv"))
content_k50 = pd.read_csv(os.path.join(results_dir, "metrics_content_LMO_k50_20251130.csv"))
collab_k50 = pd.read_csv(os.path.join(results_dir, "metrics_collaborative_k50_20251130.csv"))
hybrid_k50 = pd.read_csv(os.path.join(results_dir, "metrics_hybrid_k50_a0.2_20251130.csv"))

In [4]:
# Modelo 10

# Etiquetas modelos
popular_k10["model"] = "popularity"
content_k10["model"] = "content"
collab_k10["model"] = "collaborative"
hybrid_k10["model"] = "hybrid"

# Resumen k=10
summary_10 = pd.concat(
    [popular_k10, content_k10, collab_k10, hybrid_k10],
    ignore_index=True
)

# Save
summary_10.to_csv(os.path.join(results_dir, "summary_k10.csv"), index=False)

In [5]:
# Modelo 50

# Etiquetas modelos
popular_k50["model"] = "popularity"
content_k50["model"] = "content"
collab_k50["model"] = "collaborative"
hybrid_k50["model"] = "hybrid"

# Resumen k=50
summary_50 = pd.concat(
    [popular_k50, content_k50, collab_k50, hybrid_k50],
    ignore_index=True
)

# Save
summary_50.to_csv(os.path.join(results_dir, "summary_k50.csv"), index=False)

## Información

In [6]:
conn = sqlite3.connect(db_path)

In [7]:
# Para popularidad
games_pop = pd.read_sql("SELECT * FROM games_popular_final", conn)

In [8]:
# Para contenido / colaborativo / híbrido
games = pd.read_sql("SELECT * FROM games_escaled_final", conn)
user_train = pd.read_sql("SELECT * FROM user_train_LMO", conn)
user_test = pd.read_sql("SELECT * FROM user_test_LMO", conn)

In [9]:
conn.close()

In [10]:
# Normalización de tipos
games_pop["appid"] = games_pop["appid"].astype(np.int64)
games["appid"] = games["appid"].astype(np.int64)

user_train["user_id"] = user_train["user_id"].astype(np.int64)
user_train["appid"] = user_train["appid"].astype(np.int64)
user_test["user_id"] = user_test["user_id"].astype(np.int64)
user_test["appid"] = user_test["appid"].astype(np.int64)

In [11]:
# Artefactos modelo contenido
alpha_text = 0.7  # el mismo valor con el que la guardaste

combined_path = os.path.join(models_content_dir, f"X_combined_alpha_{alpha_text:.1f}.npz")
X_combined = sparse.load_npz(combined_path)

In [12]:
# Índice de juegos (orden compatible con X_combined)
idx = pd.read_csv(os.path.join(models_dir, "games_index.csv"))
idx["appid"] = idx["appid"].astype(np.int64)

In [13]:
# Artefactos modelo colaborativo

# Matrices usuario-juego
R = sparse.load_npz(os.path.join(models_collab_dir, "R_user_based.npz"))
R_norm = sparse.load_npz(os.path.join(models_collab_dir, "R_norm_user_based.npz"))

# Modelo kNN + mapas
knn_artifacts = joblib.load(os.path.join(models_collab_dir, "knn_user_based_70.0.pkl"))
knn = knn_artifacts["knn"]
user_map = knn_artifacts["user_map"]
game_map = knn_artifacts["game_map"]
inv_game_map = knn_artifacts["inv_game_map"]

# 📊 Resumen cuantitativo de resultados (k=10 y k=50)

In [14]:
summary_10

,k,precision@k,recall@k,f1@k,ndcg@k,hit_rate@k,item_coverage@k,num_users,model,alpha
0,10,0.024194,0.080648,0.037222,0.112910,0.219182,0.053550,47627,popularity,NaN
1,10,0.011273,0.037577,0.017343,0.057018,0.094547,0.581442,47627,content,NaN
2,10,0.072860,0.242866,0.112092,0.352222,0.542486,0.607374,47627,collaborative,NaN
3,10,0.073782,0.245939,0.113510,0.355743,0.546854,0.628849,47627,hybrid,0.2


In [15]:
summary_50

,k,precision@k,recall@k,f1@k,ndcg@k,hit_rate@k,item_coverage@k,num_users,model,alpha
0,50,0.013589,0.226475,0.025639,0.174067,0.512797,0.164300,47627,popularity,NaN
1,50,0.005353,0.089214,0.010100,0.082433,0.219980,0.888574,47627,content,NaN
2,50,0.028330,0.472169,0.053453,0.394518,0.825645,0.840357,47627,collaborative,NaN
3,50,0.028663,0.477719,0.054081,0.397572,0.829949,0.872771,47627,hybrid,0.2


# 👤 Evaluación cualitativa

## Usuario 76561198200130565

In [16]:
user_id = 76561198200130565  # usuario ejemplo
user_id = int(user_id)

In [17]:
user_test_user = user_test[user_test["user_id"] == user_id].copy()
user_test_user = user_test_user.merge(
    games[["appid", "name"]],
    on="appid",
    how="left"
)

print(user_test_user[["appid", "name"]].to_string(index=False))

  appid                    name
1097150               Fall Guys
    320 Half-Life 2: Deathmatch
1245620              ELDEN RING


In [18]:
user_train_user = user_train[user_train["user_id"] == user_id].copy()

user_train_user = user_train_user.merge(
    games[["appid", "name"]],
    on="appid",
    how="left"
)

cols = [c for c in user_train_user.columns if c != "user_id"]
print(
    user_train_user[cols]
    .to_string(index=False)
)


 appid                                 name
   220                          Half-Life 2
   360         Half-Life Deathmatch: Source
640590  The LEGO® NINJAGO® Movie Video Game
   730                     Counter-Strike 2
 70600                Worms Ultimate Mayhem
214950 Total War: ROME II - Emperor Edition
633230    NARUTO TO BORUTO: SHINOBI STRIKER
   620                             Portal 2
221100                                 DayZ
261550         Mount & Blade II: Bannerlord
222880                           Insurgency
322330                Don't Starve Together


## Sistema de recomendación por popularidad

In [19]:
recs_pop = recommend_by_popularity(
    user_id=user_id,
    games=games_pop,
    interactions_df=user_train,  # TRAIN
    k=10,
)
print("\nPOPULARIDAD: Top juegos recomendados:")
print(recs_pop[["appid", "name"]].head(10))


POPULARIDAD: Top juegos recomendados:
     appid                       name
0   271590  Grand Theft Auto V Legacy
1   105600                   Terraria
2   252490                       Rust
3   431960           Wallpaper Engine
4  2358720         Black Myth: Wukong
5   413150             Stardew Valley
6   292030   The Witcher 3: Wild Hunt
7      550              Left 4 Dead 2
8  1245620                 ELDEN RING
9  1086940            Baldur's Gate 3


## Sistema de recomendación por contenido

In [20]:
recs_content = recommend_by_content_user(
    user_id=user_id,
    user_train=user_train,
    U_norm=None,
    user_to_pos=None,
    X_combined=X_combined,
    games=games,
    idx=idx,
    top_n=10,
)

print("\nCONTENIDO: Top juegos recomendados:")
print(recs_content[["appid", "name"]].head(10))


CONTENIDO: Top juegos recomendados:
    appid                         name
0     320      Half-Life 2: Deathmatch
1      70                    Half-Life
2     240       Counter-Strike: Source
3     500                  Left 4 Dead
4  594570      Total War: WARHAMMER II
5     280            Half-Life: Source
6  252490                         Rust
7  251570                7 Days to Die
8     550                Left 4 Dead 2
9   33930  Arma 2: Operation Arrowhead


## Sistema de recomendación colaborativo

In [21]:
recs_collab = recommend_by_collaborative(
    user_id=user_id,
    knn=knn,
    R_norm=R_norm,
    R=R,
    user_map=user_map,
    inv_game_map=inv_game_map,
    idx=idx,
    top_n=10,
    top_neighbors=50,
)
print("\nCOLABORATIVO: Top juegos recomendados:")
print(recs_collab[["appid", "name"]].head(10))


COLABORATIVO: Top juegos recomendados:
    appid                       name
0     550              Left 4 Dead 2
1     320    Half-Life 2: Deathmatch
2  242760                 The Forest
3  218620                   PAYDAY 2
4  271590  Grand Theft Auto V Legacy
5  346110      ARK: Survival Evolved
6  231430        Company of Heroes 2
7  431960           Wallpaper Engine
8  105600                   Terraria
9  381210           Dead by Daylight


# Sistema de recomendación híbrido

In [22]:
alpha_hybrid = 0.2

recs_hybrid = recommend_by_hybrid(
    user_id=user_id,
    alpha=alpha_hybrid,
    user_train=user_train,
    U_norm=None,
    user_to_pos=None,
    X_combined=X_combined,
    R_norm=R_norm,
    R=R,
    user_map=user_map,
    game_map=game_map,
    inv_game_map=inv_game_map,
    games=games,
    idx=idx,
    knn=knn,
    top_n=10,
    top_neighbors=50,
)


print("\nHÍBRIDO: Top juegos recomendados:")
print(recs_hybrid[["appid", "name"]].head(10))


HÍBRIDO: Top juegos recomendados:
    appid                       name
0     550              Left 4 Dead 2
1     320    Half-Life 2: Deathmatch
2  242760                 The Forest
3  218620                   PAYDAY 2
4  271590  Grand Theft Auto V Legacy
5  346110      ARK: Survival Evolved
6     240     Counter-Strike: Source
7  252490                       Rust
8  231430        Company of Heroes 2
9      70                  Half-Life


## Usuario 76561198201929459

In [23]:
user_id = 76561198201929459  # usuario ejemplo
user_id = int(user_id)

In [24]:
user_test_user = user_test[user_test["user_id"] == user_id].copy()
user_test_user = user_test_user.merge(
    games[["appid", "name"]],
    on="appid",
    how="left"
)

print(user_test_user[["appid", "name"]].to_string(index=False))

  appid       name
 299740 Miscreated
2646460   Soulmask
 264710 Subnautica


## Sistema de recomendación por popularidad

In [25]:
recs_pop = recommend_by_popularity(
    user_id=user_id,
    games=games_pop,
    interactions_df=user_train,  # TRAIN
    k=10,
)
print("\nPOPULARIDAD: Top juegos recomendados:")
print(recs_pop[["appid", "name"]].head(10))


POPULARIDAD: Top juegos recomendados:
     appid                      name
0      730          Counter-Strike 2
1   105600                  Terraria
2   252490                      Rust
3   431960          Wallpaper Engine
4  2358720        Black Myth: Wukong
5   413150            Stardew Valley
6   292030  The Witcher 3: Wild Hunt
7      550             Left 4 Dead 2
8  1245620                ELDEN RING
9  1086940           Baldur's Gate 3


## Sistema de recomendación por contenido

In [26]:
recs_content = recommend_by_content_user(
    user_id=user_id,
    user_train=user_train,
    U_norm=None,
    user_to_pos=None,
    X_combined=X_combined,
    games=games,
    idx=idx,
    top_n=10,
)

print("\nCONTENIDO: Top juegos recomendados:")
print(recs_content[["appid", "name"]].head(10))


CONTENIDO: Top juegos recomendados:
     appid                                       name
0    12210  Grand Theft Auto IV: The Complete Edition
1   251570                              7 Days to Die
2   252490                                       Rust
3   440900                               Conan Exiles
4   299740                                 Miscreated
5  1574900                                     Salt 2
6   895400                                   Deadside
7  1766060                                   HumanitZ
8  2646460                                   Soulmask
9   593600                                     PixARK


## Sistema de recomendación colaborativo

In [27]:
recs_collab = recommend_by_collaborative(
    user_id=user_id,
    knn=knn,
    R_norm=R_norm,
    R=R,
    user_map=user_map,
    inv_game_map=inv_game_map,
    idx=idx,
    top_n=10,
    top_neighbors=50,
)
print("\nCOLABORATIVO: Top juegos recomendados:")
print(recs_collab[["appid", "name"]].head(10))


COLABORATIVO: Top juegos recomendados:
    appid                      name
0  346110     ARK: Survival Evolved
1  252490                      Rust
2     730          Counter-Strike 2
3  218620                  PAYDAY 2
4     550             Left 4 Dead 2
5  242760                The Forest
6  440900              Conan Exiles
7  251570             7 Days to Die
8  105600                  Terraria
9  306130  The Elder Scrolls Online


# Sistema de recomendación híbrido

In [28]:
alpha_hybrid = 0.2

recs_hybrid = recommend_by_hybrid(
    user_id=user_id,    alpha=alpha_hybrid,
    user_train=user_train,
    U_norm=None,    user_to_pos=None,
    X_combined=X_combined,
    R_norm=R_norm,     R=R,    user_map=user_map,
    game_map=game_map,     inv_game_map=inv_game_map,
    games=games,     idx=idx,    knn=knn,
    top_n=10,    top_neighbors=50,
)

print("\nHÍBRIDO: Top juegos recomendados:")
print(recs_hybrid[["appid", "name"]].head(10))



HÍBRIDO: Top juegos recomendados:
    appid                   name
0  346110  ARK: Survival Evolved
1  252490                   Rust
2  440900           Conan Exiles
3  218620               PAYDAY 2
4  242760             The Forest
5  251570          7 Days to Die
6     550          Left 4 Dead 2
7     730       Counter-Strike 2
8  108600        Project Zomboid
9  244850        Space Engineers
